In [25]:
import pandas as pd
knjige = pd.read_csv("knjige.csv", index_col="id")
zanri = pd.read_csv("zanri.csv")
knjige["opis"] = knjige["opis"].astype("str")
knjige.dropna(subset=["opis"])

,naslov,id_avtorja,povprecna_ocena,stevilo_ocen,stevilo_recenzij,stevilo_strani,jezik,leto_izdaje,trenutno_bere,opis
id,,,,,,,,,,
2767052,"The Hunger Games (The Hunger Games, #1)",153394.0,4.35,10080244,268807,374.0,English,2008.0,110226,Winning means fame and fortune. Losing means c...
1885,Pride and Prejudice,1265.0,4.30,4900979,153011,279.0,English,2023.0,256801,"Since its immediate success in 1813, Pride and..."
2657,To Kill a Mockingbird,1825.0,4.26,6993055,135628,323.0,English,2006.0,100313,"""Shoot all the bluejays you want, if you can h..."
58613451,Harry Potter and the Order of the Phoenix (Har...,1077326.0,4.50,3856441,80331,896.0,English,2003.0,92198,It's official: the evil Lord Voldemort has ret...
19063,The Book Thief,11466.0,4.39,2925868,163395,592.0,English,2006.0,105296,Librarian's note: An alternate cover edition c...
...,...,...,...,...,...,...,...,...,...,...
157981682,The Book of Love,24902.0,3.50,11400,2452,628.0,English,2024.0,2340,The Book of Love showcases Kelly Link at the h...
3109,The Omnivore's Dilemma: A Natural History of F...,2121.0,4.19,211330,14727,450.0,English,2006.0,11578,What should we have for dinner? For omnivore l...
264,The Portrait of a Lady,159.0,3.80,85922,4573,797.0,English,2003.0,5653,"When Isabel Archer, a beautiful, spirited Amer..."


In [ ]:
def koren_besede(beseda):
    if len(beseda) < 3 or beseda == "the" or beseda == "and":
        return ""
    
    if beseda[-3:] in ["ing"]:
        return beseda[0:-3]

    if beseda[-2:] in ["ed", "es"]:
        return beseda[0:-2]

    if beseda[-1] in ["s", "e"]:
        return beseda[0:-1]

    return beseda
    


def besede_opisa(opis):
    opis = str(opis)
    cist_opis = ""
    for crka in opis:
        if crka.isalpha() or crka == " ":
            cist_opis += crka.lower()

    besede = cist_opis.split(" ")

    besede = [koren_besede(beseda) for beseda in besede]
    besede_cisto = []
    for beseda in besede:
        if beseda != "":
            besede_cisto.append(beseda)
    
    return pd.Series(besede_cisto)


besede_opisa("Young mathemematician searches for meaning of love.")

0              young
1    mathemematician
2             search
3                for
4               mean
5                lov
dtype: str

In [31]:
knjige_besede = knjige["opis"].apply(besede_opisa)

In [35]:
knjige_besede = knjige_besede.stack().reset_index("id").rename(columns={0: "beseda"})

In [41]:
zanri_besed = pd.merge(knjige_besede, zanri, left_on="id", right_on="id_knjige")[["zanr", "beseda"]]
zanri_besed.dropna(subset=["beseda"])

,zanr,beseda
0,Young Adult,winn
1,Dystopia,winn
2,Fiction,winn
3,Fantasy,winn
4,Science Fiction,winn
...,...,...
4712498,Novels,bonhamcarter
4712499,20th Century,bonhamcarter
4712500,Romance,bonhamcarter
4712501,Audiobook,bonhamcarter


In [56]:
knjige_zanri = pd.merge(knjige, zanri, left_on="id", right_on="id_knjige").groupby("zanr").size()
verjetnost_zanra = knjige_zanri / knjige.size
knjige_zanri

zanr
18th Century                  3
19th Century                 31
20th Century                 32
21st Century                  1
Abuse                        10
                           ... 
Young Adult                 330
Young Adult Contemporary     10
Young Adult Fantasy          57
Young Adult Romance           5
Zombies                       1
Length: 344, dtype: int64

In [47]:
verjetnost_zanra

zanr
18th Century                0.0003
19th Century                0.0031
20th Century                0.0032
21st Century                0.0001
Abuse                       0.0010
                             ...  
Young Adult                 0.0330
Young Adult Contemporary    0.0010
Young Adult Fantasy         0.0057
Young Adult Romance         0.0005
Zombies                     0.0001
Length: 344, dtype: float64

In [60]:
verjetnost_besed = pd.crosstab(zanri_besed["beseda"], zanri_besed["zanr"]) / knjige_zanri + 0.00001
verjetnost_besed

zanr,18th Century,19th Century,20th Century,21st Century,Abuse,Academic,Action,Addiction,Adult,Adult Fiction,...,Witches,Womens,World War I,World War II,Writing,Young Adult,Young Adult Contemporary,Young Adult Fantasy,Young Adult Romance,Zombies
beseda,,,,,,,,,,,,,,,,,,,,,
aaron,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.000010,0.00001,...,0.00001,0.00001,0.00001,0.00001,0.00001,0.00304,0.00001,0.00001,0.00001,0.00001
ab,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.005035,0.00001,...,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001
ababa,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.000010,0.00001,...,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001
aback,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.000010,0.00001,...,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001
abagail,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.000010,0.00001,...,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
सथपत,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.000010,0.00001,...,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001
सदसय,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.000010,0.00001,...,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001
समगर,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.000010,0.00001,...,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001


In [69]:
def napovej_zanr(opis):
    besede = besede_opisa(opis)
    rezultat = verjetnost_besed[verjetnost_besed.index.isin(besede)].prod() * verjetnost_zanra
    return rezultat.sort_values(ascending=False).head(10)


opis = "Searching for the meaning of life a student builds a space ship and goes to the moon."
napovej_zanr(opis)


zanr
Science Fiction Fantasy    2.206984e-13
Adult                      1.333572e-13
Audiobook                  1.239291e-13
Book Club                  8.869973e-14
Fantasy                    8.544668e-14
Science Fiction            8.056927e-14
Fiction                    7.971282e-14
Adventure                  7.484850e-14
Novels                     2.957102e-14
Contemporary               1.955221e-14
dtype: float64

In [70]:
opis = "Student fails UVP because he cheated on the exam."
napovej_zanr(opis)

zanr
Magic           7.626855e-10
Audiobook       8.553001e-11
Fantasy         5.046885e-11
Young Adult     4.478170e-11
Fiction         2.259982e-11
Adventure       3.125386e-13
Middle Grade    2.943463e-13
Nonfiction      1.166741e-13
Childrens       1.018158e-13
Adult           7.635113e-14
dtype: float64